# Sahayak bullet extractor: LoRA fine-tune
Attach the `sahayak-finetune` dataset, set Accelerator to GPU T4 x2, Internet On, then Run All. With two GPUs both models train at the same time.

In [ ]:
!pip install -q -r $(dirname $(find /kaggle/input -name finetune_qwen.py | head -1))/requirements.txt
!pip uninstall -y -q torchao

In [ ]:
import glob, os, torch
PKG = os.path.dirname(glob.glob("/kaggle/input/**/finetune_qwen.py", recursive=True)[0])
print("package:", PKG, os.listdir(PKG))
assert torch.cuda.is_available(), "No GPU: set Session options > Accelerator to GPU T4 x2 or P100"
for i in range(torch.cuda.device_count()): print(i, torch.cuda.get_device_name(i))
print(torch.__version__)

## Train Qwen3-0.6B and Qwen3-1.7B (2 epochs each); progress is printed every 60 s

In [ ]:
import subprocess, shutil, os, time
JOBS = [("Qwen/Qwen3-0.6B", "qwen3-0.6b-bullets-lora", "--bs 4 --grad-accum 4 --grad-ckpt"),
        ("Qwen/Qwen3-1.7B", "qwen3-1.7b-bullets-lora", "--bs 4 --grad-accum 4 --grad-ckpt")]   # fp16; checkpointing keeps 1.7B inside 15 GB
n_gpu = torch.cuda.device_count()
print("GPUs:", n_gpu, "->", "training both models at the same time" if n_gpu >= 2 else "one GPU: training one after the other")

def launch(model, name, extra, gpu):
    cmd = (f"python -u {PKG}/finetune_qwen.py --train {PKG}/train.jsonl --val {PKG}/val.jsonl "
           f"--model {model} --epochs 2 --dtype fp16 --out /kaggle/working/{name} {extra}")
    log = open(f"/kaggle/working/{name}.log", "w")
    print(f"[gpu {gpu}] {cmd}")
    return subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT,
                            env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)})

def last_line(name):
    try:
        lines = [l for l in open(f"/kaggle/working/{name}.log", errors="ignore").read().splitlines() if l.strip()]
        return lines[-1][:160] if lines else "(starting...)"
    except FileNotFoundError:
        return "(no log yet)"

def wait(procs):
    while any(p.poll() is None for p in procs.values()):
        time.sleep(60)
        for name, p in procs.items():
            state = "running" if p.poll() is None else f"exit {p.poll()}"
            print(f"{time.strftime('%H:%M:%S')} {name} [{state}] {last_line(name)}")

if n_gpu >= 2:
    procs = {name: launch(m, name, extra, i) for i, (m, name, extra) in enumerate(JOBS)}
    wait(procs)
else:
    procs = {}
    for m, name, extra in JOBS:
        procs[name] = launch(m, name, extra, 0)
        wait({name: procs[name]})
for name, p in procs.items():
    print(name, "exit code", p.returncode)
failed = [n for n, p in procs.items() if p.returncode != 0]
for _, name, _ in JOBS:     # zip every model that finished (adapter + tokenizer files + train_log.json), even if the other failed
    if name not in failed and os.path.isdir(f"/kaggle/working/{name}"):
        shutil.make_archive(f"/kaggle/working/{name}", "zip", "/kaggle/working", name)
        print("wrote", f"/kaggle/working/{name}.zip", os.path.getsize(f"/kaggle/working/{name}.zip") // 1_000_000, "MB")
if failed:
    for n in failed:
        print("---- tail of", n, "----")
        print("\n".join(open(f"/kaggle/working/{n}.log", errors="ignore").read().splitlines()[-30:]))
    raise RuntimeError(f"training failed: {failed} (finished models were still zipped above)")

In [ ]:
import json
for n in ("qwen3-0.6b-bullets-lora", "qwen3-1.7b-bullets-lora"):
    p = f"/kaggle/working/{n}/train_log.json"
    if os.path.exists(p):
        l = json.load(open(p))
        print(n, "wall", l.get("wall_time_s"), "s; best epoch", l.get("best_epoch"), "val", l.get("best_val_loss"))
        for e in l["epochs_log"]: print("  ", e)
print(os.listdir("/kaggle/working"))